In [1]:
#Load Libraries
import warnings
warnings.filterwarnings("ignore")
 
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import Normalize
from scipy.interpolate import interp1d
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
 
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [2]:
N_POINTS     = 100       # points per surface (upper + lower) → input dim = 200
LATENT_DIM   = 32        # VAE bottleneck dimension
EPOCHS       = 150       # training epochs
BATCH_SIZE   = 128
LR           = 1e-3
BETA         = 1.0       # KL weight in ELBO loss (beta-VAE; 1.0 = standard VAE)
N_GENERATE   = 16        # novel airfoils to generate
SEED         = 42

torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)
 
# colour palette
C1, C2, C3, C4, C5 = "#2196F3", "#F44336", "#4CAF50", "#FF9800", "#9C27B0"
DARK   = "#0d1117"
PANEL  = "#161b22"
BORDER = "#30363d"

In [3]:
shapes  = np.load("curated_airfoils.npz")["shapes"]   # (19164, 1001, 2)
classes = np.load("curated_airfoils.npz")["classes"]  # (19164,)
print(f"  Loaded {len(shapes):,} airfoils")

  Loaded 19,164 airfoils


In [4]:
# Unified cosine-spaced x grid from 0 → 1
x_grid = (1 - np.cos(np.linspace(0, np.pi, N_POINTS))) / 2   # (N_POINTS,)

def airfoil_to_vector(shape: np.ndarray):
    """
    Convert a (1001, 2) raw airfoil array into a (2*N_POINTS,) flat vector:
      [y_upper(x_grid), y_lower(x_grid)]
 
    Returns None if interpolation fails or the shape is degenerate.
    """
    x, y   = shape[:, 0], shape[:, 1]
    le_idx = int(np.argmin(x))
 
    # split into upper (TE→LE reversed to LE→TE) and lower (LE→TE)
    x_up = x[:le_idx + 1][::-1]
    y_up = y[:le_idx + 1][::-1]
    x_lo = x[le_idx:]
    y_lo = y[le_idx:]
 
    # need monotonically increasing x for interpolation
    if not (np.all(np.diff(x_up) > 0) and np.all(np.diff(x_lo) > 0)):
        # deduplicate
        _, ui = np.unique(x_up, return_index=True)
        _, li = np.unique(x_lo, return_index=True)
        x_up, y_up = x_up[ui], y_up[ui]
        x_lo, y_lo = x_lo[li], y_lo[li]
 
    try:
        yu = np.interp(x_grid, x_up, y_up)
        yl = np.interp(x_grid, x_lo, y_lo)
    except Exception:
        return None
 
    vec = np.concatenate([yu, yl]).astype(np.float32)
 
    # basic sanity checks
    thickness = yu - yl
    if np.any(thickness < -0.01):          # lower surface above upper
        return None
    if np.max(thickness) < 0.001:          # degenerate flat plate
        return None
    if np.any(np.abs(vec) > 0.5):          # unrealistically large y values
        return None
 
    return vec

In [5]:
def vector_to_surfaces(vec: np.ndarray):
    """Reverse: split (2*N_POINTS,) vector back into upper/lower y arrays."""
    yu = vec[:N_POINTS]
    yl = vec[N_POINTS:]
    return yu, yl